In [ ]:
from __future__ import division, print_function

import ROOT

import os, sys
import math
import collections
import copy
import json

ANA_DIR = os.path.expanduser("~/nobackup/bbtautau_afresh/CMSSW_15_0_15/src/Ana")
os.chdir(ANA_DIR)
sys.path.insert(0, ANA_DIR)

ROOT.gInterpreter.AddIncludePath(ANA_DIR)  
ROOT.gROOT.LoadMacro("Particle.h+")
ROOT.gROOT.LoadMacro("HHbbtautauAnaElements.C+")

print("CWD =", os.getcwd())
print("sys.path[0] =", sys.path[0])

ROOT.gErrorIgnoreLevel = ROOT.kInfo  # show more
# ROOT.ROOT.EnableImplicitMT()         # Enable multithreading
ROOT.ROOT.DisableImplicitMT()

import anaConfig
from ROOT import Ana, TCanvas, TH1D, TPad, TLegend, THStack, RDataFrame
from argparse import ArgumentParser


In [ ]:
DATASET_DICT_PATH = "dataset_dict/"

def ls_nanoaod_files(
        year, dataset_dict_path,     
        category = "HHbbtt", 
        sample = "GluGlutoHHto2B2Tau_kl-1p00_kt-1p00_c2-0p00",
        verbose=0
    ):
    """
    Load index JSON for a given YEAR from the project root/data directory.

    Returns:
        Parsed JSON object (dict or list depending on file contents).
    """
    # Base directories
    json_file = os.path.join(dataset_dict_path, f"index_{year}.json")

    # Load JSON
    with open(json_file, "r") as f:
        data = json.load(f)
        categories = sorted(data.keys())

        if verbose:
            print(f"[{year}] Available categories:")
            for cat in categories:
                print("  -", cat)

        if category not in data:
            raise ValueError(
                f"Category '{category}' not found for {year}. "
                f"Available: {', '.join(sorted(data.keys()))}"
            )

        cat_dict = data[category]
        if sample not in cat_dict:
            raise ValueError(
                f"Sample '{sample}' not found in category '{category}' for {year}. "
                f"Available: {', '.join(sorted(cat_dict.keys()))}"
            )

        files = cat_dict[sample]
        if verbose:
            print(f"[{year}] Category '{category}', sample '{sample}':")
            print(f"  Number of files: {len(files)}")
        return files
    
def dropBranchNames(frame, filename, exclusionlist = []):
	"""
	Writes a list of column (branch) names from a ROOT RDataFrame to a text file,
	excluding any branches whose names contain substrings in the exclusion list.

	Parameters:
		frame         : ROOT.RDataFrame
			The dataframe to extract column names from.
		filename      : str
			The name of the output text file where the column names will be saved.
		exclusionlist : list of str
			List of substrings; any column containing these substrings will be excluded.
	"""
	with open(filename, "w") as file: 
		for name in frame.GetColumnNames(): 
			name = str(name)
			#print("{} {}".format(name, [(excluded in name) for excluded in exclusionlist]))
			if not (any([excluded in name for excluded in exclusionlist])): 
				file.write("{}\n".format(name))


def WriteFile(sample, filename, blacklist, treename = "Events"): 
	sample.Snapshot(treename, filename, Ana.purgeColumns(sample.GetColumnNames(), blacklist))


def generalise(df): 
	return ROOT.ROOT.RDF.AsRNode(df)


In [ ]:
YEAR = 2022
CATEGORY = "HHbbtt"
SAMPLE="GluGlutoHHto2B2Tau_kl-1p00_kt-1p00_c2-0p00",
SHORT_NAME = "ggfBoostedPrivate"

files = ls_nanoaod_files(
    year = YEAR,
    dataset_dict_path=DATASET_DICT_PATH, 
    category = "HHbbtt", 
    sample = "GluGlutoHHto2B2Tau_kl-1p00_kt-1p00_c2-0p00",
    verbose=False # True
    )
# print(files[:3])

In [ ]:
anaConfig.tauIDvar

'globalParT'

In [ ]:
def DefineTauTaggerVarsForJet(theJet, sample, tauIDvar='globalParT'): 
	sample = sample.Define("{}_{}_Xtauhtaum".format(theJet, tauIDvar), "Ana::overflowProtected(FatJet_{}_Xtauhtaum, {})".format(tauIDvar, theJet))
	sample = sample.Define("{}_{}_Xtauhtauh".format(theJet, tauIDvar), "Ana::overflowProtected(FatJet_{}_Xtauhtauh, {})".format(tauIDvar, theJet))
	sample = sample.Define("{}_{}_Xtauhtaue".format(theJet, tauIDvar), "Ana::overflowProtected(FatJet_{}_Xtauhtaue, {})".format(tauIDvar, theJet))
	return sample


def DefineBTaggerVarsForJet(theJet, sample, tauIDvar='globalParT'): 
	sample = sample.Define("{}_{}_Xbb".format(theJet, tauIDvar), "Ana::overflowProtected(FatJet_{}_Xbb, {})".format(tauIDvar, theJet))
	return sample

In [ ]:
sig = ROOT.RDataFrame("Events", files)
sig = sig.Filter("rdfentry_ >= 0 && rdfentry_ < 5000")
sig = generalise(sig)

ROOT.gStyle.SetOptStat(0)
ROOT.RDF.Experimental.AddProgressBar(sig)
print(sig)

# Creates file with the name branchnames.txt
dropBranchNames(sig, "branchnames.txt", ["L1", "HLT", "DST"])

cutflow = ROOT.CutFlow("cutflow", "Selection cutflow")

n0 = sig.Count().GetValue()
cutflow.Increment("Initial", n0)

sig = sig.Filter("nFatJet>=2").Filter("FatJet_pt[0]>250&&FatJet_pt[1]>200")

n1 = sig.Count().GetValue()
cutflow.Add("jet selection", n1)

print(n0, n1)

# sig.Display(
#     ["rdfentry_"],
#     5
# ).Print()

<cppyy.gbl.ROOT.RDF.RInterface<ROOT::Detail::RDF::RNodeBase,void> object at 0x496ae7e0>
5000 219
|======================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================

In [ ]:
# Identify an optimal combination of tagger variables to classify gen-matched bb and tautau FatJets,
# then apply the same selection to reconstructed bb and tautau-tagged FatJets in data.

gensig = sig.Define("GenDecay", "Ana::DecayGenMatching({0}_pdgId, {0}_genPartIdxMother, {0}_statusFlags)".format("GenPart"))

# Gen Fatjet (Truth-level info)
gensig = gensig.Define("GenMatchedTauFatJet", "Ana::closestMatch(GenDecay.Htotau, {0}_pt, {0}_eta, {0}_phi, {0}_mass, {1}_pt, {1}_eta, {1}_phi, {1}_mass)".format("GenPart", "FatJet"))
gensig = gensig.Define("GenMatchedBFatJet", "Ana::closestMatch(GenDecay.Htob, {0}_pt, {0}_eta, {0}_phi, {0}_mass, {1}_pt, {1}_eta, {1}_phi, {1}_mass)".format("GenPart", "FatJet"))
gensig = gensig.Define("OverlapbTauJet", "GenMatchedTauFatJet==GenMatchedBFatJet")

tauIDvar = 'globalParT'
gensig = DefineBTaggerVarsForJet("GenMatchedBFatJet", gensig, tauIDvar=tauIDvar)
gensig = DefineBTaggerVarsForJet("GenMatchedTauFatJet", gensig, tauIDvar=tauIDvar)

gensig = DefineTauTaggerVarsForJet("GenMatchedBFatJet", gensig, tauIDvar=tauIDvar)
gensig = DefineTauTaggerVarsForJet("GenMatchedTauFatJet", gensig, tauIDvar=tauIDvar)

In [109]:
# # From tauIDvar
# tauIDvar = 'globalParT'

# # Recon FatJets (Visible)
# gensig = gensig.Define(f"RecoBFatJet_{tauIDvar}", "Ana::RecoBJet( {0}_pt, {0}_eta, {0}_phi, {0}_{1})".format("FatJet", "{}_Xbb".format(tauIDvar)))
# gensig = gensig.Define(f"RecoTauFatJet_{tauIDvar}", f"1-RecoBFatJet_{tauIDvar}")

# gensig = DefineBTaggerVarsForJet(f"RecoBFatJet_{tauIDvar}", gensig, tauIDvar=tauIDvar)
# gensig = DefineBTaggerVarsForJet("GenMatchedBFatJet", gensig, tauIDvar=tauIDvar)
# gensig = DefineBTaggerVarsForJet("GenMatchedTauFatJet", gensig, tauIDvar=tauIDvar)

# gensig = DefineTauTaggerVarsForJet("GenMatchedBFatJet", gensig, tauIDvar=tauIDvar)
# gensig = DefineTauTaggerVarsForJet("GenMatchedTauFatJet", gensig, tauIDvar=tauIDvar)
# gensig = DefineTauTaggerVarsForJet(f"RecoTauFatJet_{tauIDvar}", gensig, tauIDvar=tauIDvar)

In [ ]:
blacklist = ["Muon_P4", "GenPart_Particle", "GenMuon", "HLT*", "L1*"] # TODO: add autoblacklist

# WriteFile(sig, "./SigAll.root", blacklist)
WriteFile(gensig, "./SigAllwithGen.root", blacklist)

# print("Initial: {}, 2 FatJets: {}, tauhtaumu: {}, gen mu within jet: {}, reco mu within jet: {}, other selection requirements: {}".format(n0, n1, n2, n3, n4, n5))